# Writer Agent Test Notebook

## Setup

In [ ]:
# Install required packages
# %pip install langchain langchain-openai openai python-dotenv

In [ ]:
import sys
import os

module_path = os.path.abspath(os.path.join('..'))

if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

model = os.getenv("LLM_MODEL_NAME", "gpt-4o-mini")

print(f"✅ OpenAI client initialized with model '{model}'")

In [ ]:
from utils.logging_utils import init_logs, log_openai_response, get_logs_dataframe, get_summary, list_log_files

RUN_ID = None

init_logs(RUN_ID)

print("✅ Logging initialized")
print(f"Log files: {list_log_files()}")

In [ ]:
from utils.style_manager import StyleManager

style_manager = StyleManager("../artifact/style_guides.json")

print("✅ Style Manager initialized")
print(f"BBC style keys: {list(style_manager.get_style_dict('bbc').keys())}")
print(f"Taylor Swift style keys: {list(style_manager.get_style_dict('taylor_swift').keys())}")

## Writer Function with Web Search

In [ ]:
def write_with_style(topic: str, source_type: str) -> str:

    style_dna = style_manager.get_style(source_type)

    prompt = f"""You are an AI Creative Director. Your task is to write content that matches the target source's style.

IMPORTANT: Before writing, search the web for recent information about the topic if needed.
Use the web_search tool if the topic is time-sensitive or requires current information.

## Topic
{topic}

## Style DNA to follow
{style_dna}

Write content that matches this style exactly.

Do not ask questions, write the content to the best of your ability"""

    response = client.responses.create(
        model=model,
        tools=[{"type": "web_search"}],
        input=prompt
    )

    task_name = f"{source_type}_Writer"

    output_text = log_openai_response(
        response,
        prompt,
        task_name=task_name,
        run_id=RUN_ID
    )

    return output_text


print("✅ Writer function defined with web_search support")

## Test: Write BBC-style article

In [ ]:
bbc_topic = "A new tech company has announced a breakthrough in quantum computing"
source_type = "bbc"

print(f"Topic: {bbc_topic}")
print(f"Source Type: {source_type}")

bbc_result = write_with_style(bbc_topic, source_type)

print(bbc_result)

## Test: Write Taylor Swift-style tweet

In [ ]:
ts_topic = "Taylor Swift announces her new album release date"
source_type = "taylor_swift"

print(f"Topic: {ts_topic}")
print(f"Source Type: {source_type}")

ts_result = write_with_style(ts_topic, source_type)

print(ts_result)

## Save Outputs to Artifact

In [ ]:
# Save the generated content to artifact folder
import json

corpus_data = {
    "bbc": {
        "topic": bbc_topic,
        "content": bbc_result
    },
    "taylor_swift": {
        "topic": ts_topic,
        "content": ts_result
    }
}

# Save to artifact folder
os.makedirs("../artifact", exist_ok=True)
with open("../artifact/swift_announcement_corpus.json", "w") as f:
    json.dump(corpus_data, f, indent=2)

print("✅ Saved to artifact/swift_announcement_corpus.json")

## Logging Visualization

In [ ]:
df_logs = get_logs_dataframe(RUN_ID)

display(df_logs)

summary = get_summary(RUN_ID)

print(summary)